In [2]:
import os

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\MLops\\DataScienceProject_1\\Health_premium_calculator'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataTransformationConfig:
    root_dir : Path 
    data_path : Path

In [6]:
from src.datascience.constants import * 
from src.datascience.utils.common import * 

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
        )

        return data_transformation_config

In [8]:
import os
from src.datascience import logger
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

In [19]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    
    ## Note: You can add different data transformation techniques such as Scaler, PCA and all
    #You can perform all kinds of EDA in ML cycle here before passing this data to the model

    # I am only adding train_test_spliting cz this data is already cleaned up


    def transforming_data(self):
        df = pd.read_csv(self.config.data_path) 

        ## Note-> to go through extensive EDA look at the python 
        # notebook named '03_data_transformation_eda.ipynb' int the 
        # same directory as that of this notebook.
        df.columns = df.columns.str.replace(' ', '_').str.lower() # replacing the " " in cloumn names to '_'
        # then converting them to lower case string name 

        # Dropping the null values 
        df.dropna(inplace=True)

        # No duplicare rows present 
        df.drop_duplicates(inplace=True)

        # converting the -ve dependant values to +ve 
        df['number_of_dependants'] = abs(df['number_of_dependants'])

        # treating outlier values of age and income 
        df_1 = df[df.age<=100].copy() # removing age greater than 100

        df_2 = df_1[df_1['income_lakhs']<=100] # removing income greater than 1 cr 

        # Categorical data modification on the smoking_status of our data.
        df_2['smoking_status'].replace({
            'Smoking=0':'No Smoking',
            'Does Not Smoke': 'No Smoking',
            'Not Smoking': 'No Smoking'
        }, inplace = True)

        risk_scores = {
            "diabetes": 6,
            "heart disease": 8,
            "high blood pressure":6,
            "thyroid": 5,
            "no disease": 0,
            "none":0
        }

        df_2[['disease1', 'disease2']] = df_2['medical_history'].str.split(" & ", expand=True).apply(lambda x: x.str.lower())
        df_2['disease1'].fillna('none', inplace=True)
        df_2['disease2'].fillna('none', inplace=True)
        df_2['total_risk_score'] = 0

        for disease in ['disease1', 'disease2']:
            df_2['total_risk_score'] += df_2[disease].map(risk_scores)

        # Normalize the risk score to a range of 0 to 1
        max_score = df_2['total_risk_score'].max()
        min_score = df_2['total_risk_score'].min()
        df_2['normalized_risk_score'] = (df_2['total_risk_score'] - min_score) / (max_score - min_score)

        df_2['insurance_plan'] = df_2['insurance_plan'].map({'Bronze': 1, 'Silver': 2, 'Gold': 3})

        df_2['income_level'] = df_2['income_level'].map({'<10L':1, '10L - 25L': 2, '25L - 40L':3, '> 40L':4})

        nominal_cols = ['gender', 'region', 'marital_status', 'bmi_category', 'smoking_status', 'employment_status']
        df_3 = pd.get_dummies(df_2, columns=nominal_cols, drop_first=True, dtype=int)

        df_4 = df_3.drop(['medical_history','disease1', 'disease2', 'total_risk_score',], axis=1)

        cols_to_scale = ['age','number_of_dependants','income_level', 'income_lakhs', 'insurance_plan']
        scaler = MinMaxScaler()

        df_4[cols_to_scale] = scaler.fit_transform(df_4[cols_to_scale])

        df_4.drop('income_level', axis='columns', inplace=True)

        train, test = train_test_split(df_4)

        train.to_csv(os.path.join(self.config.root_dir, "train.csv"),index = False)
        test.to_csv(os.path.join(self.config.root_dir, "test.csv"),index = False)

        logger.info("Splited data into training and test sets")
        logger.info(train.shape)
        logger.info(test.shape)

        print(train.shape)
        print(test.shape)

        

In [20]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.transforming_data()
except Exception as e:
    raise e

[2025-05-01 13:37:54,222:INFO:common:yaml file: config\config.yaml loaded successfully]
[2025-05-01 13:37:54,224:INFO:common:yaml file: params.yaml loaded successfully]
[2025-05-01 13:37:54,230:INFO:common:yaml file: schema.yaml loaded successfully]
artifacts already exists, skipping.
artifacts/data_tranformation already exists, skipping.


C:\Users\Ujjwal\AppData\Local\Temp\ipykernel_6924\205359557.py:36: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_2['smoking_status'].replace({
C:\Users\Ujjwal\AppData\Local\Temp\ipykernel_6924\205359557.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2['smoking_status'].replace({
C:\Users\Ujjwal\AppData\Local\Temp\ipykernel_6924\205

[2025-05-01 13:37:55,628:INFO:205359557:Splited data into training and test sets]
[2025-05-01 13:37:55,628:INFO:205359557:(37431, 18)]
[2025-05-01 13:37:55,635:INFO:205359557:(12477, 18)]
(37431, 18)
(12477, 18)
